In [2]:
# =============================================================================
# 🚀 MARS LANDSLIDE SEGMENTATION - COMPETITION WINNING SOLUTION
# Target: Beat mIoU 0.79 (current #1)
# Strategy: 3-Model Ensemble + TTA + Optimal Loss + Per-Band Normalization
# =============================================================================

# ─────────────────────────────────────────────────────────────────────────────
# CELL 1: Install dependencies
# ─────────────────────────────────────────────────────────────────────────────
!pip install -q segmentation-models-pytorch==0.3.4 albumentations rasterio ttach

# ─────────────────────────────────────────────────────────────────────────────
# CELL 2: Imports
# ─────────────────────────────────────────────────────────────────────────────
import os
import gc
import random
import zipfile
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler

import albumentations as A
from albumentations.pytorch import ToTensorV2

import segmentation_models_pytorch as smp
import rasterio
import ttach

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.8/58.8 kB 3.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.8/58.8 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.5/109.5 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 36.9 MB/s eta 0:00:00a 0:00:01


In [61]:
# ============================================================
# CELL 3 — Paths
# ============================================================
import glob
from pathlib import Path

BASE = '/kaggle/input/datasets/shradhanjali15/mars-ls'

TRAIN_IMG_DIR  = f'{BASE}/train/images'
TRAIN_MASK_DIR = f'{BASE}/train/masks'
VAL_IMG_DIR    = f'{BASE}/val/images'
VAL_MASK_DIR   = f'{BASE}/val/masks'

# Check what test images you actually have
TEST_IMG_DIR   = f'{BASE}/test/images'

train_imgs  = sorted(glob.glob(f'{TRAIN_IMG_DIR}/*.tif'))
train_masks = sorted(glob.glob(f'{TRAIN_MASK_DIR}/*.tif'))
val_imgs    = sorted(glob.glob(f'{VAL_IMG_DIR}/*.tif'))
val_masks   = sorted(glob.glob(f'{VAL_MASK_DIR}/*.tif'))
test_imgs   = sorted(glob.glob(f'{TEST_IMG_DIR}/*.tif'))

# Phase-2 expected filenames from Codabench GT
PHASE2_NAMES = [
    'col00000_row00010.tif','col00000_row00011.tif','col00001_row00009.tif',
    'col00001_row00010.tif','col00001_row00011.tif','col00002_row00008.tif',
    'col00002_row00009.tif','col00003_row00005.tif','col00003_row00006.tif',
    'col00003_row00008.tif','col00003_row00013.tif','col00004_row00005.tif',
    'col00004_row00006.tif','col00004_row00008.tif','col00004_row00012.tif',
    'col00004_row00013.tif','col00005_row00006.tif','col00005_row00007.tif',
    'col00005_row00008.tif','col00005_row00009.tif','col00005_row00010.tif',
    'col00006_row00005.tif','col00006_row00006.tif','col00006_row00007.tif',
    'col00006_row00008.tif','col00006_row00009.tif','col00006_row00010.tif',
    'col00006_row00011.tif','col00007_row00005.tif','col00007_row00006.tif',
    'col00007_row00007.tif','col00007_row00008.tif','col00007_row00010.tif',
    'col00007_row00011.tif','col00008_row00005.tif','col00008_row00006.tif',
    'col00008_row00008.tif','col00008_row00009.tif','col00009_row00005.tif',
    'col00009_row00006.tif','col00009_row00007.tif','col00009_row00009.tif',
    'col00009_row00010.tif','col00009_row00011.tif','col00010_row00004.tif',
    'col00010_row00005.tif','col00010_row00008.tif','col00010_row00009.tif',
    'col00010_row00010.tif','col00010_row00011.tif','col00011_row00004.tif',
    'col00011_row00005.tif','col00011_row00009.tif','col00011_row00010.tif',
    'col00011_row00011.tif','col00012_row00004.tif','col00012_row00005.tif',
    'col00012_row00006.tif','col00012_row00009.tif','col00012_row00010.tif',
    'col00013_row00005.tif','col00013_row00006.tif','col00013_row00007.tif',
    'col00013_row00008.tif','col00013_row00009.tif','col00013_row00010.tif',
    'col00013_row00011.tif','col00014_row00005.tif','col00014_row00006.tif',
    'col00014_row00007.tif','col00014_row00008.tif','col00014_row00009.tif',
    'col00014_row00010.tif','col00015_row00005.tif','col00015_row00006.tif',
    'col00015_row00007.tif','col00015_row00008.tif','col00015_row00009.tif',
    'col00015_row00010.tif','col00015_row00011.tif','col00016_row00005.tif',
    'col00016_row00006.tif','col00016_row00007.tif','col00016_row00008.tif',
    'col00016_row00009.tif','col00016_row00010.tif','col00016_row00011.tif',
    'col00017_row00004.tif','col00017_row00005.tif','col00017_row00006.tif',
    'col00017_row00007.tif','col00017_row00008.tif','col00017_row00009.tif',
    'col00017_row00010.tif','col00017_row00011.tif','col00017_row00012.tif',
    'col00018_row00005.tif','col00018_row00006.tif','col00018_row00007.tif',
    'col00018_row00008.tif','col00018_row00009.tif','col00018_row00010.tif',
    'col00018_row00011.tif','col00018_row00012.tif','col00018_row00013.tif',
    'col00019_row00005.tif','col00019_row00006.tif','col00019_row00007.tif',
    'col00019_row00008.tif','col00019_row00009.tif','col00019_row00010.tif',
    'col00019_row00011.tif','col00019_row00012.tif','col00019_row00013.tif',
    'col00019_row00015.tif','col00019_row00016.tif','col00019_row00017.tif',
    'col00020_row00002.tif','col00020_row00003.tif','col00020_row00004.tif',
    'col00020_row00005.tif','col00020_row00006.tif','col00020_row00007.tif',
    'col00020_row00008.tif','col00020_row00009.tif','col00020_row00010.tif',
    'col00020_row00011.tif','col00020_row00012.tif','col00020_row00013.tif',
    'col00020_row00014.tif','col00020_row00015.tif','col00020_row00016.tif',
    'col00020_row00017.tif','col00020_row00018.tif','col00021_row00000.tif',
    'col00021_row00001.tif','col00021_row00002.tif','col00021_row00003.tif',
    'col00021_row00006.tif','col00021_row00007.tif','col00021_row00008.tif',
    'col00021_row00009.tif','col00021_row00010.tif','col00021_row00011.tif',
    'col00021_row00012.tif','col00021_row00015.tif','col00021_row00016.tif',
    'col00021_row00017.tif','col00021_row00018.tif','col00022_row00007.tif',
    'col00022_row00008.tif','col00022_row00009.tif','col00022_row00010.tif',
    'col00022_row00011.tif','col00022_row00012.tif','col00022_row00013.tif',
    'col00022_row00014.tif','col00022_row00015.tif','col00022_row00018.tif',
    'col00022_row00019.tif','col00022_row00020.tif','col00022_row00021.tif',
    'col00023_row00006.tif','col00023_row00007.tif','col00023_row00008.tif',
    'col00023_row00009.tif','col00023_row00010.tif','col00023_row00011.tif',
    'col00023_row00012.tif','col00023_row00013.tif','col00023_row00014.tif',
    'col00023_row00015.tif','col00023_row00016.tif','col00023_row00017.tif',
    'col00023_row00018.tif','col00023_row00019.tif','col00023_row00020.tif',
    'col00023_row00021.tif','col00023_row00022.tif','col00024_row00004.tif',
    'col00024_row00005.tif','col00024_row00006.tif','col00024_row00007.tif',
    'col00024_row00008.tif','col00024_row00009.tif','col00024_row00010.tif',
    'col00024_row00011.tif','col00024_row00012.tif','col00024_row00013.tif',
    'col00024_row00014.tif','col00024_row00015.tif','col00024_row00017.tif',
    'col00024_row00018.tif','col00024_row00019.tif','col00024_row00020.tif',
    'col00024_row00021.tif','col00024_row00022.tif','col00024_row00023.tif',
    'col00025_row00004.tif','col00025_row00005.tif','col00025_row00006.tif',
    'col00025_row00007.tif','col00025_row00008.tif','col00025_row00009.tif',
    'col00025_row00010.tif','col00025_row00011.tif','col00025_row00012.tif',
    'col00025_row00013.tif','col00025_row00014.tif','col00025_row00015.tif',
    'col00025_row00016.tif','col00025_row00017.tif','col00025_row00018.tif',
    'col00025_row00019.tif','col00025_row00020.tif','col00025_row00021.tif',
    'col00026_row00005.tif','col00026_row00006.tif','col00026_row00007.tif',
    'col00026_row00008.tif','col00026_row00009.tif','col00026_row00010.tif',
    'col00026_row00011.tif','col00026_row00012.tif','col00026_row00013.tif',
    'col00026_row00014.tif','col00026_row00015.tif','col00026_row00016.tif',
    'col00026_row00017.tif','col00027_row00005.tif','col00027_row00006.tif',
    'col00027_row00007.tif','col00027_row00008.tif','col00027_row00009.tif',
    'col00027_row00011.tif','col00027_row00012.tif','col00027_row00013.tif',
    'col00027_row00014.tif','col00027_row00015.tif','col00027_row00016.tif',
    'col00028_row00003.tif','col00028_row00004.tif','col00028_row00005.tif',
    'col00028_row00006.tif','col00028_row00007.tif','col00028_row00008.tif',
    'col00028_row00009.tif','col00028_row00011.tif','col00028_row00012.tif',
    'col00028_row00013.tif','col00028_row00014.tif','col00029_row00002.tif',
    'col00029_row00003.tif','col00029_row00004.tif','col00029_row00005.tif',
    'col00029_row00006.tif','col00029_row00007.tif','col00029_row00008.tif',
    'col00029_row00011.tif','col00029_row00012.tif','col00029_row00013.tif',
    'col00029_row00014.tif','col00030_row00002.tif','col00030_row00003.tif',
    'col00030_row00004.tif','col00030_row00005.tif','col00030_row00012.tif',
    'col00030_row00013.tif','col00030_row00014.tif','col00031_row00002.tif',
    'col00031_row00003.tif','col00031_row00004.tif','col00031_row00005.tif',
    'col00031_row00012.tif','col00031_row00013.tif','col00031_row00014.tif',
]

print(f"Train images : {len(train_imgs)}")
print(f"Val   images : {len(val_imgs)}")
print(f"Test  images (local): {len(test_imgs)}")
print(f"Phase-2 expected    : {len(PHASE2_NAMES)}")

# Check if phase-2 files exist in our test folder
test_name_to_path = {Path(p).name: p for p in test_imgs}
phase2_found = [n for n in PHASE2_NAMES if n in test_name_to_path]
print(f"Phase-2 files found in test folder: {len(phase2_found)} / {len(PHASE2_NAMES)}")

Train images : 465
Val   images : 66
Test  images (local): 133
Phase-2 expected    : 276
Phase-2 files found in test folder: 46 / 276


In [30]:
# ============================================================
# CELL 4 — Seed & Config
# ============================================================
def seed_everything(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(42)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

CFG = {
    'output_dir'  : '/kaggle/working',
    'epochs'      : 120,
    'batch_size'  : 32,
    'val_batch'   : 64,
    'num_workers' : 0,       # FIXED: 0 for Kaggle
    'lr'          : 3e-4,
    'weight_decay': 1e-4,
    'patience'    : 20,
    'dice_weight' : 0.6,
    'bce_weight'  : 0.4,
    'use_amp'     : False,   # FIXED: disable AMP — was causing NaN
    'use_tta'     : True,
    'models': [
        ('unetplusplus',  'efficientnet-b4', 'imagenet'),
        ('unet',          'resnet50',        'imagenet'),   # FIXED: resnet50 supports 7ch
        ('deeplabv3plus', 'resnet34',        'imagenet'),   # FIXED: lighter, faster
    ],
    'seeds': [42, 123, 2024],
}

Device: cuda
GPU: Tesla P100-PCIE-16GB
Memory: 17.1 GB


In [31]:
# ============================================================
# CELL 5 — Per-band normalization (train set only!)
# ============================================================
def compute_band_stats_robust(img_paths, num_bands=7):
    print("Computing robust per-band stats...")
    s1 = np.zeros(num_bands, dtype=np.float64)
    s2 = np.zeros(num_bands, dtype=np.float64)
    n  = 0
    for p in tqdm(img_paths):
        with rasterio.open(p) as src:
            img = src.read().astype(np.float64)
        img = np.nan_to_num(img, nan=0.0, posinf=0.0, neginf=0.0)
        s1 += img.sum(axis=(1,2))
        s2 += (img**2).sum(axis=(1,2))
        n  += img.shape[1] * img.shape[2]
    mean = s1 / n
    std  = np.sqrt(np.maximum(s2/n - mean**2, 1e-8))
    std  = np.where((std > 1e6) | (std < 1e-6), 1.0, std)
    print(f"mean: {mean.round(3)}")
    print(f"std : {std.round(3)}")
    return mean.astype(np.float32), std.astype(np.float32)

BAND_MEAN, BAND_STD = compute_band_stats_robust(train_imgs)

Computing robust per-band stats...


  0%|          | 0/465 [00:00<?, ?it/s]

mean: [ 1.24918000e+02 -3.43026559e+34  8.99369000e+02  1.25896000e+02
  1.16216000e+02  9.02300000e+01  8.77360000e+01]
std : [3.879300e+01 1.000000e+00 3.125313e+03 2.071900e+01 4.395700e+01
 5.057400e+01 5.589800e+01]


In [41]:
# ============================================================
# CELL 6 — Dataset
# ============================================================
# ============================================================
# CELL 6 — Dataset
# ============================================================
class MarsDataset(Dataset):
    def __init__(self, img_paths, mask_paths, transform=None, is_test=False):
        self.img_paths = img_paths
        self.mask_paths = mask_paths
        self.transform = transform
        self.is_test = is_test

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, idx):
        with rasterio.open(self.img_paths[idx]) as src:
            img = src.read().astype(np.float32)

        # Clean NaN/Inf before normalization
        img = np.nan_to_num(img, nan=0.0, posinf=0.0, neginf=0.0)
        img = (img - BAND_MEAN[:, None, None]) / BAND_STD[:, None, None]
        img = np.clip(img, -5.0, 5.0).transpose(1, 2, 0)  # (H, W, 7)

        if not self.is_test:
            with rasterio.open(self.mask_paths[idx]) as src:
                mask = src.read(1).astype(np.float32)

            mask = np.nan_to_num(mask, nan=0.0, posinf=0.0, neginf=0.0)
            mask = (mask > 0.5).astype(np.float32)  # exact binary {0,1}

            if self.transform:
                out = self.transform(image=img, mask=mask)
                img, mask = out["image"], out["mask"]
            else:
                img = torch.from_numpy(img.transpose(2, 0, 1))
                mask = torch.from_numpy(mask)

            return img, mask.unsqueeze(0)

        else:
            if self.transform:
                img = self.transform(image=img)["image"]
            else:
                img = torch.from_numpy(img.transpose(2, 0, 1))

            return img, Path(self.img_paths[idx]).name

In [42]:
# ============================================================
# CELL 7 — Augmentations
# ============================================================
def get_train_transforms():
    return A.Compose([
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.RandomRotate90(p=0.5),
        A.Transpose(p=0.3),

        A.OneOf([
            A.ElasticTransform(alpha=10, sigma=3, p=1.0),
            A.GridDistortion(num_steps=5, distort_limit=0.2, p=1.0),
        ], p=0.20),

        A.GaussNoise(var_limit=(1e-4, 5e-3), p=0.15),

        A.CoarseDropout(
            max_holes=6,
            max_height=12,
            max_width=12,
            min_holes=1,
            min_height=4,
            min_width=4,
            fill_value=0,
            p=0.20
        ),

        ToTensorV2(),
    ])

In [ ]:
# ============================================================
# CELL 7B — compute positive class weight
# ============================================================


def compute_pos_weight(mask_paths):
    pos = 0
    total = 0
    print("Computing positive class weight...")
    for p in tqdm(mask_paths):
        with rasterio.open(p) as src:
            m = src.read(1).astype(np.float32)
        m = np.nan_to_num(m, nan=0.0, posinf=0.0, neginf=0.0)
        m = (m > 0.5).astype(np.float32)
        pos += m.sum()
        total += m.size

    neg = total - pos
    pos_weight = neg / max(pos, 1.0)
    pos_weight = float(np.clip(pos_weight, 1.0, 8.0))  # keep stable
    print(f"POS_WEIGHT = {pos_weight:.4f}")
    return pos_weight

POS_WEIGHT = compute_pos_weight(train_masks)

In [ ]:
# ============================================================
# CELL 8 — Loss
# ============================================================
class DiceBCELoss(nn.Module):
    def __init__(self, dw=0.6, bw=0.4, smooth=1.0, pos_weight=1.0):
        super().__init__()
        self.dw = dw
        self.bw = bw
        self.smooth = smooth
        self.pos_weight = float(pos_weight)

    def forward(self, logits, targets):
        targets = targets.float()

        # weighted BCE
        bce = F.binary_cross_entropy_with_logits(
            logits,
            targets,
            pos_weight=torch.tensor([self.pos_weight], device=logits.device)
        )

        # soft dice
        probs = torch.sigmoid(logits)
        probs = probs.view(probs.size(0), -1)
        t = targets.view(targets.size(0), -1)

        inter = (probs * t).sum(dim=1)
        denom = probs.sum(dim=1) + t.sum(dim=1)
        dice_loss = 1.0 - ((2.0 * inter + self.smooth) / (denom + self.smooth)).mean()

        return self.dw * dice_loss + self.bw * bce

In [45]:
# ============================================================
# CELL 9 — Metrics
# ============================================================
def compute_metrics(pred_bin, targets):
    p=pred_bin.float().view(-1); t=targets.float().view(-1)
    tp=(p*t).sum(); fp=(p*(1-t)).sum(); fn=((1-p)*t).sum(); tn=((1-p)*(1-t)).sum()
    iou_fg=tp/(tp+fp+fn+1e-6); iou_bg=tn/(tn+fp+fn+1e-6)
    pre=tp/(tp+fp+1e-6); rec=tp/(tp+fn+1e-6); f1=2*pre*rec/(pre+rec+1e-6)
    return {'miou':((iou_fg+iou_bg)/2).item(),'iou_fg':iou_fg.item(),
            'iou_bg':iou_bg.item(),'precision':pre.item(),'recall':rec.item(),'f1':f1.item()}

In [46]:
# ============================================================
# CELL 10 — Model factory
# ============================================================
def build_model(arch, encoder, enc_w):
    kw = dict(encoder_name=encoder, encoder_weights=enc_w,
              in_channels=7, classes=1, activation=None)
    m = {'unetplusplus':smp.UnetPlusPlus,'unet':smp.Unet,
         'deeplabv3plus':smp.DeepLabV3Plus,'fpn':smp.FPN}[arch](**kw)
    return m.to(DEVICE)

In [47]:
# ============================================================
# CELL 11 — Train / Val loops
# ============================================================
@torch.no_grad()
def val_epoch(model, loader, crit):
    model.eval()
    total = 0.0
    pa, ma = [], []

    for imgs, masks in tqdm(loader, desc='Val  ', leave=False):
        imgs = imgs.to(DEVICE)
        masks = masks.to(DEVICE).float()

        with autocast(enabled=CFG['use_amp']):
            logits = model(imgs)
            loss = crit(logits, masks)

        if not torch.isfinite(logits).all():
            raise RuntimeError("Non-finite logits in validation")
        if not torch.isfinite(loss):
            raise RuntimeError("Validation loss became NaN/Inf")

        total += loss.item()
        probs = torch.sigmoid(logits)
        pa.append((probs > 0.5).long().cpu())
        ma.append(masks.long().cpu())

    return total / len(loader), compute_metrics(torch.cat(pa), torch.cat(ma))

In [48]:
# ============================================================
# CELL 12 — Train one model
# ============================================================
def train_model(idx, arch, encoder, enc_w, seed):
    print(f"\n{'='*55}\nModel {idx+1}: {arch} + {encoder}  seed={seed}\n{'='*55}")
    seed_everything(seed)
    tds = MarsDataset(train_imgs, train_masks, get_train_transforms())
    vds = MarsDataset(val_imgs,   val_masks,   get_val_transforms())
    tl  = DataLoader(tds, CFG['batch_size'], shuffle=True,
                     num_workers=CFG['num_workers'], pin_memory=True, drop_last=True)
    vl  = DataLoader(vds, CFG['val_batch'],  shuffle=False,
                     num_workers=CFG['num_workers'], pin_memory=True)

    model  = build_model(arch, encoder, enc_w)
    
    crit = DiceBCELoss(
    CFG['dice_weight'],
    CFG['bce_weight'],
    smooth=1.0,
    pos_weight=POS_WEIGHT
    )
    
    opt    = torch.optim.AdamW(model.parameters(), lr=CFG['lr'], weight_decay=CFG['weight_decay'])
    sched  = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(opt, T_0=30, T_mult=2, eta_min=1e-6)
    scaler = GradScaler(enabled=CFG['use_amp'])

    best_miou=0.0; pat=0; sp=f"{CFG['output_dir']}/best_{idx}.pth"
    for ep in range(1, CFG['epochs']+1):
        tl_loss=train_epoch(model,tl,opt,crit,scaler)
        vl_loss,met=val_epoch(model,vl,crit)
        sched.step(); miou=met['miou']
        print(f"Ep{ep:03d} TrL={tl_loss:.4f} VlL={vl_loss:.4f} "
              f"mIoU={miou:.4f} fg={met['iou_fg']:.4f} F1={met['f1']:.4f} "
              f"LR={sched.get_last_lr()[0]:.1e}")
        if miou > best_miou:
            best_miou=miou; pat=0; torch.save(model.state_dict(), sp)
            print(f"  NEW BEST: {best_miou:.4f}")
        else:
            pat+=1
            if pat >= CFG['patience']: print(f"  Early stop ep{ep}"); break

    model.load_state_dict(torch.load(sp, map_location=DEVICE)); model.eval()
    del tds,vds,tl,vl,opt,scaler; gc.collect(); torch.cuda.empty_cache()
    print(f"Model {idx+1} best mIoU: {best_miou:.4f}")
    return model, best_miou


In [50]:
# ============================================================
# CELL 13 — Train all 3
# ============================================================
trained_models=[]; best_scores=[]
for i,(arch,enc,encw) in enumerate(CFG['models']):
    m,s = train_model(i, arch, enc, encw, CFG['seeds'][i])
    trained_models.append(m); best_scores.append(s)


Model 1: unetplusplus + efficientnet-b4  seed=42


Train:   0%|          | 0/14 [00:00<?, ?it/s]

Val  :   0%|          | 0/2 [00:00<?, ?it/s]

Ep001 TrL=0.6715 VlL=0.6169 mIoU=0.5544 fg=0.3820 F1=0.5528 LR=3.0e-04
  NEW BEST: 0.5544


Train:   0%|          | 0/14 [00:00<?, ?it/s]

...

Val  :   0%|          | 0/2 [00:00<?, ?it/s]

Ep044 TrL=0.2209 VlL=0.3618 mIoU=0.7691 fg=0.7041 F1=0.8264 LR=2.6e-04


Train:   0%|          | 0/14 [00:00<?, ?it/s]

Val  :   0%|          | 0/2 [00:00<?, ?it/s]

Ep045 TrL=0.2119 VlL=0.2807 mIoU=0.7811 fg=0.7092 F1=0.8299 LR=2.6e-04
  Early stop ep45
Model 3 best mIoU: 0.7999


In [52]:
# ============================================================
# CELL 14 — Find optimal threshold
# ============================================================
@torch.no_grad()
def find_threshold(models):
    print("\nFinding optimal threshold...")
    vds=MarsDataset(val_imgs,val_masks,get_val_transforms())
    vl=DataLoader(vds,CFG['val_batch'],shuffle=False,num_workers=CFG['num_workers'])
    ap,am=[],[]
    for imgs,masks in tqdm(vl):
        imgs=imgs.to(DEVICE)
        probs=torch.zeros(imgs.shape[0],1,128,128,device=DEVICE)
        for m in models:
            with autocast(enabled=CFG['use_amp']):
                probs+=torch.sigmoid(m(imgs))
        probs/=len(models); ap.append(probs.cpu()); am.append(masks.cpu())
    ap=torch.cat(ap); am=torch.cat(am).long()
    bt,bm=0.5,0.0
    for t in np.arange(0.30,0.71,0.02):
        met=compute_metrics((ap>t).long(),am)
        print(f"  t={t:.2f} mIoU={met['miou']:.4f} F1={met['f1']:.4f} fg={met['iou_fg']:.4f}")
        if met['miou']>bm: bm,bt=met['miou'],t
    print(f"\nBest t={bt:.2f} mIoU={bm:.4f}")
    return float(bt), bm

BEST_T, ENS_MIOU = find_threshold(trained_models)


Finding optimal threshold...


  0%|          | 0/2 [00:00<?, ?it/s]

  t=0.30 mIoU=0.7879 F1=0.8447 fg=0.7312
  t=0.32 mIoU=0.7933 F1=0.8485 fg=0.7369
  t=0.34 mIoU=0.8006 F1=0.8537 fg=0.7447
  t=0.36 mIoU=0.8067 F1=0.8580 fg=0.7513
  t=0.38 mIoU=0.8106 F1=0.8606 fg=0.7553
  t=0.40 mIoU=0.8135 F1=0.8625 fg=0.7583
  t=0.42 mIoU=0.8157 F1=0.8639 fg=0.7605
  t=0.44 mIoU=0.8176 F1=0.8650 fg=0.7622
  t=0.46 mIoU=0.8192 F1=0.8660 fg=0.7637
  t=0.48 mIoU=0.8206 F1=0.8668 fg=0.7649
  t=0.50 mIoU=0.8216 F1=0.8673 fg=0.7657
  t=0.52 mIoU=0.8227 F1=0.8678 fg=0.7665
  t=0.54 mIoU=0.8236 F1=0.8682 fg=0.7671
  t=0.56 mIoU=0.8243 F1=0.8684 fg=0.7674
  t=0.58 mIoU=0.8249 F1=0.8685 fg=0.7676
  t=0.60 mIoU=0.8251 F1=0.8683 fg=0.7673
  t=0.62 mIoU=0.8251 F1=0.8679 fg=0.7666
  t=0.64 mIoU=0.8242 F1=0.8667 fg=0.7647
  t=0.66 mIoU=0.8232 F1=0.8654 fg=0.7628
  t=0.68 mIoU=0.8221 F1=0.8641 fg=0.7607
  t=0.70 mIoU=0.8209 F1=0.8627 fg=0.7585

Best t=0.60 mIoU=0.8251


In [62]:
# ============================================================
# CELL 15 — Predict test masks
# ============================================================
@torch.no_grad()
def predict_test(models, threshold=0.60):
    ds = MarsDataset(test_imgs, mask_paths=None, transform=get_val_transforms(), is_test=True)
    dl = DataLoader(
        ds,
        batch_size=CFG['val_batch'],
        shuffle=False,
        num_workers=CFG['num_workers'],
        pin_memory=True
    )

    preds = {}

    for imgs, names in tqdm(dl, desc='Predict'):
        imgs = imgs.to(DEVICE)

        with autocast(enabled=CFG['use_amp']):
            probs = torch.zeros(
                (imgs.size(0), 1, imgs.size(2), imgs.size(3)),
                device=DEVICE
            )

            for m in models:
                m.eval()
                logits = m(imgs)
                probs += torch.sigmoid(logits)

            probs /= len(models)

        probs = torch.nan_to_num(probs, nan=0.0, posinf=1.0, neginf=0.0)
        masks = (probs > threshold).squeeze(1).to(torch.uint8).cpu().numpy()

        for name, mask in zip(names, masks):
            preds[name] = mask

    print(f"Created predictions for {len(preds)} test images")
    return preds

preds = predict_test(trained_models, threshold=BEST_T)

Predict:   0%|          | 0/3 [00:00<?, ?it/s]

Created predictions for 133 test images


In [63]:
# ============================================================
# CELL 16 — Save submission.zip
# ============================================================
def save_submission(preds, ref_path):
    assert len(preds) == len(test_imgs), f"Expected {len(test_imgs)} predictions, got {len(preds)}"

    md = Path(CFG['output_dir']) / 'pred_masks'
    md.mkdir(exist_ok=True)

    # Clear old files so stale masks do not get zipped
    for f in md.glob('*.tif'):
        f.unlink()

    with rasterio.open(ref_path) as ref:
        profile = ref.profile.copy()

    profile.update(
        dtype='uint8',
        count=1,
        compress='lzw'
    )

    for fn, mask in tqdm(preds.items(), desc='Saving'):
        mask = (mask > 0).astype(np.uint8)  # force exact binary {0,1}

        with rasterio.open(md / fn, 'w', **profile) as dst:
            dst.write(mask[np.newaxis, :, :])

    zp = Path(CFG['output_dir']) / 'submission.zip'
    if zp.exists():
        zp.unlink()

    with zipfile.ZipFile(zp, 'w', zipfile.ZIP_DEFLATED) as zf:
        for f in sorted(md.glob('*.tif')):
            zf.write(f, arcname=f.name)

    with zipfile.ZipFile(zp) as zf:
        names = zf.namelist()

    print(f"submission.zip: {len(names)} files  path={zp}")
    print(f"Sample: {names[:3]}")
    return zp

sub_path = save_submission(preds, test_imgs[0])

Saving:   0%|          | 0/133 [00:00<?, ?it/s]

submission.zip: 133 files  path=/kaggle/working/submission.zip
Sample: ['col00000_row00013.tif', 'col00002_row00006.tif', 'col00002_row00007.tif']


In [64]:
import os
from pathlib import Path

# Scan ALL available input files
print("=== SCANNING ALL KAGGLE INPUT ===")
for root, dirs, files in os.walk('/kaggle/input'):
    tifs = [f for f in files if f.endswith('.tif')]
    if tifs:
        print(f"\n{root}: {len(tifs)} tif files")
        print("  First 3:", tifs[:3])

print("\n=== ALSO CHECK WORKING DIR ===")
for root, dirs, files in os.walk('/kaggle/working'):
    if files:
        print(f"{root}: {files[:5]}")

=== SCANNING ALL KAGGLE INPUT ===

/kaggle/input/datasets/shradhanjali15/mars-ls/val/images: 66 tif files
  First 3: ['col00076_row00023.tif', 'col00030_row00015.tif', 'col00018_row00005.tif']

/kaggle/input/datasets/shradhanjali15/mars-ls/val/masks: 66 tif files
  First 3: ['col00076_row00023.tif', 'col00030_row00015.tif', 'col00018_row00005.tif']

/kaggle/input/datasets/shradhanjali15/mars-ls/test/images: 133 tif files
  First 3: ['col00000_row00013.tif', 'col00016_row00008.tif', 'col00006_row00009.tif']

/kaggle/input/datasets/shradhanjali15/mars-ls/train/images: 465 tif files
  First 3: ['col00010_row00004.tif', 'col00044_row00022.tif', 'col00029_row00013.tif']

/kaggle/input/datasets/shradhanjali15/mars-ls/train/masks: 465 tif files
  First 3: ['col00010_row00004.tif', 'col00044_row00022.tif', 'col00029_row00013.tif']

=== ALSO CHECK WORKING DIR ===
/kaggle/working: ['best_0.pth', 'submission.zip', 'best_2.pth', 'best_1.pth']
/kaggle/working/pred_masks: ['col00039_row00020.tif', '